In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
from lsff_utils import paths, gbd_data
import pathlib

In [2]:
location = "India"
vehicle = "rice"
data_source = paths.data_source(default="local")  # or a model number on the team drive

In [3]:
# Parameters
location = "india"
vehicle = "salt"


In [4]:
location = location.title()

In [5]:
# Simulation output lives in per-run, timestamp-named directories, so the run
# is resolved rather than hardcoded -- this is the same run the Snakefile's
# marker names, so the notebook and the workflow cannot disagree.
maternal_results = paths.latest_results(
    paths.source_root(paths.MATERNAL_RESULTS_ROOT, data_source), location, vehicle
)
maternal_results

PosixPath('/mnt/team/simulation_science/pub/models/vivarium_gates_lsff_2026/results/model2.6.4/maternal/salt/india/2026_11_23_02_26_57/results')

In [6]:
with gbd_data.quiet_gbd_logs():
    pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2023        2024        1.936945e+05
                  0.019178   0.076712    2023        2024        5.766332e+05
                  0.076712   0.500000    2023        2024        4.257480e+06
                  0.500000   1.000000    2023        2024        5.021249e+06
                  1.000000   2.000000    2023        2024        1.024770e+07
                  2.000000   5.000000    2023        2024        3.231858e+07
                  5.000000   10.000000   2023        2024        5.835119e+07
                  10.000000  15.000000   2023        2024        6.286521e+07
                  15.000000  20.000000   2023        2024        6.432311e+07
                  20.000000  25.000000   2023        2024        6.374038e+07
                  25.000000  30.000000   2023        2024        6.159864e+07
                  30.000000  35.000000   2023        2024        5.733795e+07
  

In [7]:
with gbd_data.quiet_gbd_logs():
    asfr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
    ).value
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end  parameter  
India     Female  10.0       15.0     2023        2024      lower_value    0.000159
                                                            mean_value     0.000349
                                                            upper_value    0.000705
                  15.0       20.0     2023        2024      lower_value    0.007324
                                                            mean_value     0.008912
                                                            upper_value    0.010701
                  20.0       25.0     2023        2024      lower_value    0.080282
                                                            mean_value     0.094863
                                                            upper_value    0.111437
                  25.0       30.0     2023        2024      lower_value    0.113628
                                                            mean_value     0.124175
    

In [8]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

location  sex     age_start  age_end     year_start  year_end
India     Female  0.000000   0.019178    2023        2024        0.000000
                  0.019178   0.076712    2023        2024        0.000000
                  0.076712   0.500000    2023        2024        0.000000
                  0.500000   1.000000    2023        2024        0.000000
                  1.000000   2.000000    2023        2024        0.000000
                  2.000000   5.000000    2023        2024        0.000000
                  5.000000   10.000000   2023        2024        0.000000
                  10.000000  15.000000   2023        2024        0.000349
                  15.000000  20.000000   2023        2024        0.008912
                  20.000000  25.000000   2023        2024        0.094863
                  25.000000  30.000000   2023        2024        0.124175
                  30.000000  35.000000   2023        2024        0.079277
                  35.000000  40.000000   2023     

In [9]:
with gbd_data.quiet_gbd_logs():
    sbr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.stillbirth_28_weeks_to_live_birth_ratio, "estimate", location
    ).value
sbr[sbr > 0]

location  year_start  year_end  parameter  
India     2023        2024      lower_value    0.014030
                                mean_value     0.016334
                                upper_value    0.019572
Name: value, dtype: float64

In [10]:
sbr = sbr.iloc[0]
sbr

np.float64(0.0140296131060656)

In [11]:
with gbd_data.quiet_gbd_logs():
    maternal_abortion_miscarriage_incidence = vivarium_inputs.get_measure(
        gbd_mapping.causes.maternal_abortion_and_miscarriage,
        "incidence_rate",
        location,
    ).mean(axis=1)
maternal_abortion_miscarriage_incidence[maternal_abortion_miscarriage_incidence > 0]

quiet_gbd_logs: 6 data-quality warning(s) from vivarium_inputs, 3 distinct:
      2x  Data returned extra columns: {'gbd_process_version_id'}.
      2x  Data contains additional age groups beyond those specified by restriction range.
      2x  Data is missing the following expected sex ids: {np.int64(1)}.
  Set gbd_data.DEFAULT_VERBOSITY = gbd_data.Verbosity.WARNINGS to see these live, or Verbosity.QUIET to stop reporting them.


location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2023        2024        0.000091
                  15.0       20.0     2023        2024        0.001445
                  20.0       25.0     2023        2024        0.009951
                  25.0       30.0     2023        2024        0.010996
                  30.0       35.0     2023        2024        0.007060
                  35.0       40.0     2023        2024        0.003508
                  40.0       45.0     2023        2024        0.002148
                  45.0       50.0     2023        2024        0.000981
                  50.0       55.0     2023        2024        0.000181
dtype: float64

In [12]:
with gbd_data.quiet_gbd_logs():
    ectopic_pregnancy_incidence = vivarium_inputs.get_measure(
        gbd_mapping.causes.ectopic_pregnancy,
        "incidence_rate",
        location,
    ).mean(axis=1)
ectopic_pregnancy_incidence[ectopic_pregnancy_incidence > 0]

quiet_gbd_logs: 6 data-quality warning(s) from vivarium_inputs, 3 distinct:
      2x  Data returned extra columns: {'gbd_process_version_id'}.
      2x  Data contains additional age groups beyond those specified by restriction range.
      2x  Data is missing the following expected sex ids: {np.int64(1)}.
  Set gbd_data.DEFAULT_VERBOSITY = gbd_data.Verbosity.WARNINGS to see these live, or Verbosity.QUIET to stop reporting them.


location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2023        2024        0.000027
                  15.0       20.0     2023        2024        0.000164
                  20.0       25.0     2023        2024        0.001235
                  25.0       30.0     2023        2024        0.001726
                  30.0       35.0     2023        2024        0.001269
                  35.0       40.0     2023        2024        0.000639
                  40.0       45.0     2023        2024        0.000388
                  45.0       50.0     2023        2024        0.000176
                  50.0       55.0     2023        2024        0.000053
dtype: float64

In [13]:
pregnancy_incidence = (
    asfr
    + (asfr * sbr)
    + maternal_abortion_miscarriage_incidence
    + ectopic_pregnancy_incidence
)
pregnancy_incidence[pregnancy_incidence > 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2023        2024        0.000472
                  15.0       20.0     2023        2024        0.010646
                  20.0       25.0     2023        2024        0.107379
                  25.0       30.0     2023        2024        0.138639
                  30.0       35.0     2023        2024        0.088718
                  35.0       40.0     2023        2024        0.037709
                  40.0       45.0     2023        2024        0.014287
                  45.0       50.0     2023        2024        0.004205
                  50.0       55.0     2023        2024        0.000511
dtype: float64

In [14]:
pregnancies = pop * pregnancy_incidence
pregnancies[pregnancies > 0]

location  sex     age_start  age_end  year_start  year_end
India     Female  10.0       15.0     2023        2024        2.965630e+04
                  15.0       20.0     2023        2024        6.847552e+05
                  20.0       25.0     2023        2024        6.844406e+06
                  25.0       30.0     2023        2024        8.539952e+06
                  30.0       35.0     2023        2024        5.086922e+06
                  35.0       40.0     2023        2024        2.005770e+06
                  40.0       45.0     2023        2024        6.691576e+05
                  45.0       50.0     2023        2024        1.680654e+05
                  50.0       55.0     2023        2024        1.780321e+04
dtype: float64

In [15]:
total_pregnancies = pregnancies.sum()
total_pregnancies

np.float64(24046488.820088055)

In [16]:
sim_population = (
    pd.read_parquet(
        paths.measure_path(maternal_results, "pregnancy_outcome_count")
    )
    .groupby(["input_draw", "scenario"])
    .value.sum()
    .mean()
)
sim_population

np.float64(10000000.0)

In [17]:
scalar = total_pregnancies / sim_population
scalar

np.float64(2.4046488820088054)

In [18]:
for result in [
    "ylds",
    "ylls",
    "pregnancy_outcome_count",
    "person_time_anemia",
    "transition_count_maternal_disorders",
]:
    df = pd.read_parquet(
        paths.measure_path(maternal_results, result)
    )
    df.value *= scalar
    path = (
        paths.output_root(paths.RESCALED_PREGNANCY_RESULTS_ROOT)
        / vehicle
        / location.lower()
        / f"{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)